# CRN_AD — Scan Analysis

Load and explore the CSV produced by `scan.py`.

```bash
python scan.py \
  --n_species 4 \
  --target_pH 9 5 7 \
  --eval_pKa 8.0 7.0 6.0 5.0 \
  --eval_phi 0.05 0.1 0.2 \
  --eval_J   2.0 3.0 3.5 \
  --duration 20 30 \
  --outdir   scan_outputs
```


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import ast

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Load CSV

In [ ]:
CSV_PATH = 'scan_outputs/scan_results.csv'   # adjust if needed

df = pd.read_csv(CSV_PATH)

# Parse the 'pKa' column from string repr back to list
df['pKa'] = df['pKa'].apply(ast.literal_eval)

print(f'Loaded {len(df)} rows')
df.head()

## 2. Summary statistics

In [ ]:
print('Column dtypes')
print(df.dtypes)
print()

for col in ['target_score', 'best_other_score', 'selectivity']:
    print(f'{col}:  mean={df[col].mean():.4f}  '
          f'std={df[col].std():.4f}  '
          f'min={df[col].min():.4f}  '
          f'max={df[col].max():.4f}')

## 3. Which parameter combinations give the highest target score?

In [ ]:
df.sort_values('target_score', ascending=False).head(15)

## 4. Which give the best selectivity (target − best other)?

In [ ]:
df.sort_values('selectivity', ascending=False).head(15)

## 5. Target score vs individual parameters

One subplot per numerical sweep axis.  Points are coloured by selectivity.

In [ ]:
# Identify columns that actually vary
num_cols = ['phi', 'J', 'S_max', 'duration', 'equil_duration', 'beta', 'k0', 'n_types']
varying  = [c for c in num_cols if df[c].nunique() > 1]

if not varying:
    print('No numeric parameters vary.')
else:
    ncols = min(3, len(varying))
    nrows = (len(varying) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5*ncols, 4*nrows), squeeze=False)
    fig.suptitle('Target score vs swept parameters', fontsize=13)

    vmin, vmax = df['selectivity'].min(), df['selectivity'].max()
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    cmap = cm.RdYlGn

    for idx, col in enumerate(varying):
        ax = axes[idx // ncols][idx % ncols]
        sc = ax.scatter(df[col], df['target_score'],
                        c=df['selectivity'], cmap=cmap, norm=norm,
                        edgecolors='none', s=25, alpha=0.8)
        ax.set_xlabel(col, fontsize=11)
        ax.set_ylabel('target score', fontsize=11)
        ax.set_ylim(-0.02, 1.02)
        ax.axhline(0.5, color='grey', lw=0.7, ls='--')
        ax.grid(True, alpha=0.3)

    for idx in range(len(varying), nrows*ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap),
                 ax=axes.ravel().tolist(), label='selectivity', shrink=0.6)
    plt.tight_layout()
    plt.show()

## 6. 2-D heatmap: J vs phi (if both vary)

In [ ]:
if df['J'].nunique() > 1 and df['phi'].nunique() > 1:
    pivot = df.groupby(['J', 'phi'])['target_score'].mean().unstack()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, col, title in zip(
            axes,
            ['target_score', 'selectivity'],
            ['Mean target score', 'Mean selectivity']):
        piv = df.groupby(['J', 'phi'])[col].mean().unstack()
        im  = ax.imshow(piv.values, aspect='auto', origin='lower',
                        cmap='viridis' if col == 'target_score' else 'RdYlGn',
                        vmin=0 if col == 'target_score' else -0.5,
                        vmax=1 if col == 'target_score' else  0.5)
        ax.set_xticks(range(len(piv.columns)))
        ax.set_xticklabels([f'{v:.2f}' for v in piv.columns])
        ax.set_yticks(range(len(piv.index)))
        ax.set_yticklabels([f'{v:.2f}' for v in piv.index])
        ax.set_xlabel('phi')
        ax.set_ylabel('J (kT)')
        ax.set_title(title)
        plt.colorbar(im, ax=ax)

    plt.tight_layout()
    plt.show()
else:
    print('J or phi does not vary — skipping heatmap.')

## 7. Distribution of scores

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col, color in zip(
        axes,
        ['target_score', 'best_other_score', 'selectivity'],
        ['#27ae60', '#e74c3c', '#3498db']):
    ax.hist(df[col], bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(df[col].mean(), color='black', lw=1.5, ls='--', label='mean')
    ax.set_xlabel(col)
    ax.set_ylabel('count')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Custom query

Filter and explore subsets of the results.

In [ ]:
# Example: high-selectivity combinations with target score > 0.6
mask = (df['target_score'] > 0.6) & (df['selectivity'] > 0.2)
df[mask].sort_values('selectivity', ascending=False)